In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


In [2]:
import os
csv_path = 'demand_forecasting.csv'
if not os.path.exists(csv_path):
    csv_path = '/kaggle/input/datasets/raminhuseyn/demand-forecasting-dataset/demand_forecasting.csv'
df = pd.read_csv(csv_path)
df.head(3)


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,Weather Condition,Promotion,Competitor Pricing,Seasonality,Epidemic,Demand
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,Snowy,0,85.73,Winter,0,115
1,2022-01-01,S001,P0002,Clothing,North,117,117,249,80.16,15,Snowy,1,92.02,Winter,0,229
2,2022-01-01,S001,P0003,Clothing,North,247,114,612,62.94,10,Snowy,1,60.08,Winter,0,157


In [3]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76000 entries, 0 to 75999
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                76000 non-null  object 
 1   Store ID            76000 non-null  object 
 2   Product ID          76000 non-null  object 
 3   Category            76000 non-null  object 
 4   Region              76000 non-null  object 
 5   Inventory Level     76000 non-null  int64  
 6   Units Sold          76000 non-null  int64  
 7   Units Ordered       76000 non-null  int64  
 8   Price               76000 non-null  float64
 9   Discount            76000 non-null  int64  
 10  Weather Condition   76000 non-null  object 
 11  Promotion           76000 non-null  int64  
 12  Competitor Pricing  76000 non-null  float64
 13  Seasonality         76000 non-null  object 
 14  Epidemic            76000 non-null  int64  
 15  Demand              76000 non-null  int64  
dtypes: f

In [4]:
df.isnull().sum().sum()


np.int64(0)

In [5]:
df.duplicated().sum()


np.int64(0)

In [6]:
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')

In [7]:
import holidays

df['Date'] = pd.to_datetime(df['Date'])

# Sort by series and date to compute lags and rolling features correctly
df = df.sort_values(by=['Store ID', 'Product ID', 'Date']).reset_index(drop=True)

# Lags of Target
df['lag_1'] = df.groupby(['Store ID', 'Product ID'])['Demand'].shift(1)
df['lag_7'] = df.groupby(['Store ID', 'Product ID'])['Demand'].shift(7)
df['lag_30'] = df.groupby(['Store ID', 'Product ID'])['Demand'].shift(30)

# Rolling window statistics (shift by 1 day first to prevent data leakage)
df['rolling_mean_7'] = df.groupby(['Store ID', 'Product ID'])['Demand'].transform(lambda x: x.shift(1).rolling(7).mean())
df['rolling_std_7'] = df.groupby(['Store ID', 'Product ID'])['Demand'].transform(lambda x: x.shift(1).rolling(7).std())

df['rolling_mean_14'] = df.groupby(['Store ID', 'Product ID'])['Demand'].transform(lambda x: x.shift(1).rolling(14).mean())
df['rolling_std_14'] = df.groupby(['Store ID', 'Product ID'])['Demand'].transform(lambda x: x.shift(1).rolling(14).std())

df['rolling_mean_30'] = df.groupby(['Store ID', 'Product ID'])['Demand'].transform(lambda x: x.shift(1).rolling(30).mean())
df['rolling_std_30'] = df.groupby(['Store ID', 'Product ID'])['Demand'].transform(lambda x: x.shift(1).rolling(30).std())

# Day of week and weekend features
df['day_of_week'] = df['Date'].dt.dayofweek
df['is_weekend'] = df['Date'].dt.dayofweek.isin([5, 6]).astype(int)

# Indian holidays feature
in_holidays = holidays.country_holidays('IN')
df['is_holiday'] = df['Date'].apply(lambda x: 1 if x in in_holidays else 0)

# Trend feature (days since start date)
start_date = df['Date'].min()
df['trend'] = (df['Date'] - start_date).dt.days

# Pricing & supply features from original pipeline
df['Total_Earnings'] = df['Units Sold'] * df['Price'] * (1 - df['Discount']/100)
df['Products_to_sell'] = df['Units Ordered'] - df['Units Sold']

df['Price_gap_pct'] = (df['Price'] - df['Competitor Pricing']) / df['Competitor Pricing']
df['Discount_effect'] = df['Discount'] * df['Promotion']

# Inventory features — use inventory-to-demand ratio and days-of-supply instead of
# Inventory / (Inventory + 1), which saturates near 1 for any level above ~50.
df['inventory_demand_ratio'] = df['Inventory Level'] / (df['rolling_mean_7'] + 1e-6)
df['days_of_supply'] = df['Inventory Level'] / (df['rolling_mean_30'] + 1e-6)

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day

# Drop rows with NaNs resulting from shifts and rolling windows (first 30 days)
df = df.dropna().reset_index(drop=True)

# Sort primarily by Date and secondarily by Store ID and Product ID for chronological validation
df = df.sort_values(by=['Date', 'Store ID', 'Product ID']).reset_index(drop=True)


In [8]:
numeric_cols = df.select_dtypes(include="number").columns
object_cols = df.select_dtypes(include="object").columns

print("🔢 NUMERIC COLUMNS")
print("-" * 50)

for col in numeric_cols:
    nunique = df[col].nunique()
    print(f"\n📌 {col}")
    print(f"nunique: {nunique}")
    
    if nunique < 15:
        print("unique values:", df[col].unique())

print("\n" + "=" * 60)

print("🔤 OBJECT COLUMNS")
print("-" * 50)

for col in object_cols:
    nunique = df[col].nunique()
    print(f"\n📌 {col}")
    print(f"nunique: {nunique}")
    
    if nunique < 15:
        print("unique values:", df[col].unique())


🔢 NUMERIC COLUMNS
--------------------------------------------------

📌 Inventory Level
nunique: 1414

📌 Units Sold
nunique: 329

📌 Units Ordered
nunique: 993

📌 Price
nunique: 15288

📌 Discount
nunique: 6
unique values: [10  5  0 15 20 25]

📌 Promotion
nunique: 2
unique values: [0 1]

📌 Competitor Pricing
nunique: 15836

📌 Epidemic
nunique: 2
unique values: [0 1]

📌 Demand
nunique: 339

📌 lag_1
nunique: 339

📌 lag_7
nunique: 339

📌 lag_30
nunique: 340

📌 rolling_mean_7
nunique: 1346

📌 rolling_std_7
nunique: 71339

📌 rolling_mean_14
nunique: 2290

📌 rolling_std_14
nunique: 72161

📌 rolling_mean_30
nunique: 4152

📌 rolling_std_30
nunique: 72358

📌 day_of_week
nunique: 7
unique values: [0 1 2 3 4 5 6]

📌 is_weekend
nunique: 2
unique values: [0 1]

📌 is_holiday
nunique: 2
unique values: [0 1]

📌 trend
nunique: 730

📌 Total_Earnings
nunique: 70480

📌 Products_to_sell
nunique: 1137

📌 Price_gap_pct
nunique: 72482

📌 Discount_effect
nunique: 5
unique values: [ 0 15 10 20 25]

📌 inventory_de

<# d# i# v#  # s# t# y# l# e# =# "# 
b# a# c# k# g# r# o# u# n# d# :#  # l# i# n# e# a# r# -# g# r# a# d# i# e# n# t# (# 1# 3# 5# d# e# g# ,#  # ## 0# 2# 0# 6# 1# 7#  # 0# %# ,#  # ## 0# 2# 0# 6# 1# 7#  # 1# 0# 0# %# )# ;# 
b# o# r# d# e# r# -# r# a# d# i# u# s# :#  # 1# 4# p# x# ;# 
p# a# d# d# i# n# g# :#  # 1# 8# p# x#  # 2# 4# p# x# ;# 
m# a# r# g# i# n# :#  # 2# 8# p# x#  # 0# ;# 
b# o# x# -# s# h# a# d# o# w# :#  # 0#  # 1# 2# p# x#  # 3# 0# p# x#  # r# g# b# a# (# 0# ,# 0# ,# 0# ,# 0# .# 6# )# ;# 
f# o# n# t# -# f# a# m# i# l# y# :#  # A# r# i# a# l# ,#  # H# e# l# v# e# t# i# c# a# ,#  # s# a# n# s# -# s# e# r# i# f# ;# 
"# ># 

<# h# 2#  # s# t# y# l# e# =# "# 
m# a# r# g# i# n# :#  # 0# ;# 
f# o# n# t# -# s# i# z# e# :#  # 2# 6# p# x# ;# 
f# o# n# t# -# w# e# i# g# h# t# :#  # 7# 0# 0# ;# 
c# o# l# o# r# :#  # ## e# 5# e# 7# e# b# ;# 
l# e# t# t# e# r# -# s# p# a# c# i# n# g# :#  # 0# .# 6# p# x# ;# 
"# ># 
📊#  # E# D# A#  # &#  # V# i# s# u# a# l# i# z# a# t# i# o# n# 
<# /# h# 2# ># 

<# /# d# i# v# >


In [9]:
plt.figure(figsize=(14,6))

sns.lineplot(data=df, x='Date', y='Units Sold', label='Units Sold', color='red') 
sns.lineplot(data=df, x='Date', y='Units Ordered', label='Units Ordered', color='#0D47A1')  

plt.title("Units Sold vs Units Ordered Over Time", fontsize=14)
plt.xlabel("Date")
plt.ylabel("Units")
plt.grid(alpha=0.2)

plt.legend()
plt.tight_layout()
plt.show()


In [10]:
plt.figure(figsize=(10,6))

sns.scatterplot(
    data=df,
    x='Price_gap_pct',
    y='Demand',
    hue='Promotion',
    alpha=0.6
)

plt.title("Demand vs Price Gap (Competitive Effect)")
plt.grid(alpha=0.2)
plt.show()


In [11]:
cols_for_count = ['Category','Region','Weather Condition','Seasonality','Discount','Promotion','Epidemic']
sns.set_style("darkgrid")

fig, axes = plt.subplots(4, 2, figsize=(14, 12))
axes = axes.flatten()

for i, col in enumerate(cols_for_count):
    ax = sns.countplot(
        data=df,
        x=col,
        ax=axes[i],
        palette="Set2")
    
    axes[i].set_title(f"{col} Distribution", fontsize=12, weight="bold")
    axes[i].tick_params(axis='x', rotation=30)

    for p in ax.patches:
        height = p.get_height()
        ax.annotate(
            f"{int(height)}",
            (p.get_x() + p.get_width() / 2., height),
            ha="center",
            va="bottom",
            fontsize=9)

if len(cols_for_count) < len(axes):
    for j in range(len(cols_for_count), len(axes)):
        fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


In [12]:
categorical_cols = ['Category', 'Region', 'Weather Condition', 'Seasonality']

plt.style.use("dark_background")

fig, axes = plt.subplots(2, 2, figsize=(14,10))
axes = axes.flatten()  

for i, col in enumerate(categorical_cols):
    
    n_colors = df[col].nunique()
    palette = sns.color_palette("husl", n_colors)
    
    sns.scatterplot(
        data=df,
        x='Units Sold',
        y='Demand',
        hue=col,
        palette=palette,
        alpha=0.7,
        ax=axes[i]
    )
    
    axes[i].set_title(f"{col}")
    axes[i].grid(alpha=0.2)
    
    axes[i].legend(title=col, fontsize=8)

plt.tight_layout()
plt.show()


In [13]:
plt.style.use("dark_background")

fig, axes = plt.subplots(1, 2, figsize=(16,6))

ct_cat = pd.crosstab(df['Category'], df['Discount'])
ct_cat = ct_cat.sort_index(axis=1)
ct_cat_pct = ct_cat.div(ct_cat.sum(axis=1), axis=0) * 100

ax1 = ct_cat_pct.plot(
    kind='bar',
    stacked=True,
    colormap='viridis',
    ax=axes[0]
)

for container in ax1.containers:
    ax1.bar_label(container, fmt='%.1f%%', label_type='center', fontsize=8)

ax1.set_title("Discount Distribution by Category (%)")
ax1.set_xlabel("Category")
ax1.set_ylabel("Percentage")
ax1.legend(title="Discount", fontsize=8)


ct_season = pd.crosstab(df['Seasonality'], df['Discount'])
ct_season = ct_season.sort_index(axis=1)

order = ['Winter', 'Spring', 'Summer', 'Autumn']
ct_season = ct_season.loc[order]

ct_season_pct = ct_season.div(ct_season.sum(axis=1), axis=0) * 100

ax2 = ct_season_pct.plot(
    kind='bar',
    stacked=True,
    colormap='viridis',
    ax=axes[1]
)

for container in ax2.containers:
    ax2.bar_label(container, fmt='%.1f%%', label_type='center', fontsize=8)

ax2.set_title("Discount Distribution by Season (%)")
ax2.set_xlabel("Season")
ax2.set_ylabel("Percentage")
ax2.legend(title="Discount", fontsize=8)


plt.tight_layout()
plt.show()


In [14]:
plt.figure(figsize=(8,5))

season_avg = df.groupby('Seasonality')['Total_Earnings'].mean().sort_values()

ax = sns.barplot(
    x=season_avg.index,
    y=season_avg.values,
    palette="viridis"
)

for container in ax.containers:
    ax.bar_label(container, fmt='%.0f', label_type='center')

plt.title("Average Total Earnings by Season")
plt.ylabel("Avg Earnings")

plt.tight_layout()
plt.show()


In [15]:
plt.figure(figsize=(14,6))

sns.lineplot(
    data=df,
    x='Date',
    y='Total_Earnings',
    hue='Seasonality',
    palette='husl'
)

plt.title("Total Earnings Over Time by Season")
plt.grid(alpha=0.2)

plt.tight_layout()
plt.show()


In [16]:
plt.figure(figsize=(8,5))

sns.histplot(df['Products_to_sell'], bins=50, kde=True, color='orange')

plt.title("Products_to_sell Distribution")
plt.grid(alpha=0.2)

plt.show()


In [17]:
plt.figure(figsize=(8,5))

sns.scatterplot(
    data=df,
    x='Demand',
    y='Products_to_sell',
    hue='Seasonality',
    alpha=0.6
)

plt.title("Demand vs Products to Sell (Stock Gap Analysis)")
plt.xlabel("Demand")
plt.ylabel("Products to Sell")

plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()


In [18]:
plt.style.use("default")

corr = df.select_dtypes(include='number').corr()

mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(10,8))

sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="viridis",
    center=0,
    linewidths=0.5
)

plt.title("Correlation Heatmap (Lower Triangle)")
plt.tight_layout()
plt.show()


In [19]:
df_model = df.copy()

df_model = df_model.drop(columns=[
    'Units Sold',
    'Units Ordered',
    'Products_to_sell',
    'Total_Earnings',
    'Price',
    'Date'
])


<# d# i# v#  # s# t# y# l# e# =# "# 
b# a# c# k# g# r# o# u# n# d# :#  # l# i# n# e# a# r# -# g# r# a# d# i# e# n# t# (# 1# 3# 5# d# e# g# ,#  # ## 0# 2# 0# 6# 1# 7#  # 0# %# ,#  # ## 0# 2# 0# 6# 1# 7#  # 1# 0# 0# %# )# ;# 
b# o# r# d# e# r# -# r# a# d# i# u# s# :#  # 1# 4# p# x# ;# 
p# a# d# d# i# n# g# :#  # 1# 8# p# x#  # 2# 4# p# x# ;# 
m# a# r# g# i# n# :#  # 2# 8# p# x#  # 0# ;# 
b# o# x# -# s# h# a# d# o# w# :#  # 0#  # 1# 2# p# x#  # 3# 0# p# x#  # r# g# b# a# (# 0# ,# 0# ,# 0# ,# 0# .# 6# )# ;# 
f# o# n# t# -# f# a# m# i# l# y# :#  # A# r# i# a# l# ,#  # H# e# l# v# e# t# i# c# a# ,#  # s# a# n# s# -# s# e# r# i# f# ;# 
"# ># 

<# h# 2#  # s# t# y# l# e# =# "# 
m# a# r# g# i# n# :#  # 0# ;# 
f# o# n# t# -# s# i# z# e# :#  # 2# 6# p# x# ;# 
f# o# n# t# -# w# e# i# g# h# t# :#  # 7# 0# 0# ;# 
c# o# l# o# r# :#  # ## e# 5# e# 7# e# b# ;# 
l# e# t# t# e# r# -# s# p# a# c# i# n# g# :#  # 0# .# 6# p# x# ;# 
"# ># 
🤖#  # M# o# d# e# l# i# n# g#  # &#  # P# r# e# d# i# c# t# i# o# n# 
<# /# h# 2# ># 

<# /# d# i# v# >


In [20]:
# Split train and test chronologically by holding out the last 12 weeks (84 days) of the dataset
max_date = df['Date'].max()
split_date = max_date - pd.Timedelta(days=84)

train_mask = df['Date'] <= split_date
test_mask = df['Date'] > split_date

y = np.log1p(df_model['Demand'])
X = df_model.drop(columns=['Demand'])

X_train = X[train_mask]
X_test = X[test_mask]
y_train = y[train_mask]
y_test = y[test_mask]

cat_cols = ['Store ID', 'Product ID', 'Category', 'Region', 'Weather Condition', 'Seasonality']

# Note: TargetEncoder is placed inside the pipeline to avoid data leakage during CV folds.


In [21]:
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from category_encoders import TargetEncoder
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

tscv = TimeSeriesSplit(n_splits=5)

# Pipeline combining target encoding and LightGBM regressor
lgb_pipeline = Pipeline([
    ('encoder', TargetEncoder(cols=cat_cols, smoothing=0.3)),
    ('regressor', LGBMRegressor(
        random_state=42,
        reg_alpha=0.1,
        reg_lambda=0.1,
        verbose=-1
    ))
])

# Pipeline combining target encoding and XGBoost regressor
xgb_pipeline = Pipeline([
    ('encoder', TargetEncoder(cols=cat_cols, smoothing=0.3)),
    ('regressor', XGBRegressor(
        random_state=42,
        reg_alpha=0.1,
        reg_lambda=0.1
    ))
])

lgb_params = {
    'regressor__n_estimators': [500, 800, 1200, 1500],
    'regressor__learning_rate': [0.01, 0.03, 0.05],
    'regressor__num_leaves': [31, 50, 70, 100],
    'regressor__max_depth': [-1, 5, 10],
    'regressor__min_child_samples': [10, 20, 30],
    'regressor__subsample': [0.7, 0.8],
    'regressor__colsample_bytree': [0.7, 0.8],
}

xgb_params = {
    'regressor__n_estimators': [500, 800, 1200, 1500],
    'regressor__learning_rate': [0.01, 0.03, 0.05],
    'regressor__max_depth': [4, 6, 8],
    'regressor__min_child_weight': [1, 3, 5],
    'regressor__subsample': [0.7, 0.8],
    'regressor__colsample_bytree': [0.7, 0.8],
}

lgb_search = RandomizedSearchCV(
    lgb_pipeline,
    lgb_params,
    n_iter=10,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    n_jobs=1,
    verbose=1,
    random_state=42
)

xgb_search = RandomizedSearchCV(
    xgb_pipeline,
    xgb_params,
    n_iter=10,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    n_jobs=1,
    verbose=1,
    random_state=42
)


lgb_search.fit(X_train, y_train)
xgb_search.fit(X_train, y_train)

lgb_best = lgb_search.best_estimator_
xgb_best = xgb_search.best_estimator_


Fitting 5 folds for each of 10 candidates, totalling 50 fits


Fitting 5 folds for each of 10 candidates, totalling 50 fits


In [22]:
from sklearn.metrics import mean_absolute_error, r2_score
results = []

def evaluate_model(name, model):
    preds = np.expm1(model.predict(X_test))
    y_true = np.expm1(y_test)
    
    mae = mean_absolute_error(y_true, preds)
    r2 = r2_score(y_true, preds)
    
    results.append({
        "Model": name,
        "MAE": mae,
        "R2": r2
    })

evaluate_model("LightGBM", lgb_best)
evaluate_model("XGBoost", xgb_best)

results_df = pd.DataFrame(results)
print(results_df)


      Model        MAE        R2
0  LightGBM   8.094364  0.922330
1   XGBoost  10.664318  0.883048


In [23]:
plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
ax1 = sns.barplot(data=results_df, x="Model", y="R2", palette="viridis")
plt.title("R2 Comparison")

for p in ax1.patches:
    height = p.get_height()
    ax1.annotate(f"{height:.3f}",
                 (p.get_x() + p.get_width() / 2., height),
                 ha='center', va='bottom')

plt.subplot(1,2,2)
ax2 = sns.barplot(data=results_df, x="Model", y="MAE", palette="magma")
plt.title("MAE Comparison")

for p in ax2.patches:
    height = p.get_height()
    ax2.annotate(f"{height:.2f}",
                 (p.get_x() + p.get_width() / 2., height),
                 ha='center', va='bottom')

plt.tight_layout()
plt.show()


In [24]:
lgb_pred = np.expm1(lgb_best.predict(X_test))
xgb_pred = np.expm1(xgb_best.predict(X_test))

final_pred = 0.5 * lgb_pred + 0.5 * xgb_pred

print("ENSEMBLE")
print("MAE:", mean_absolute_error(np.expm1(y_test), final_pred))
print("R2:", r2_score(np.expm1(y_test), final_pred))


ENSEMBLE
MAE: 8.817210135495484
R2: 0.9126440207560883
